In [ ]:
"""
download_external_references_to_ttps.py

From enterprise-attack.json:
- Find all external_references for attack-pattern (technique) objects
- Skip MITRE domains
- Download new resources and extract text (PDF/HTML/TXT)
- Map them to the technique (TTP)
"""

import os
import json
import time
import re
import hashlib
import mimetypes
from datetime import datetime
from urllib.parse import urlparse
import requests
from requests.adapters import HTTPAdapter, Retry
from pdfminer.high_level import extract_text as extract_pdf_text
import trafilatura

# ---------------- Config ----------------
INPUT_FILE = "enterprise-attack.json"
OUT_FILE = "external_links_to_ttps.json"
DOWNLOAD_DIR = "downloads_external_refs"
REQUEST_DELAY = 0
TIMEOUT = 20
MAX_SIZE_BYTES = 15 * 1024**2
USER_AGENT = "ExternalTTPDownloader/1.0 (+https://your-domain.example/)"
# ----------------------------------------

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=[500,502,503,504])
session.mount("http://", HTTPAdapter(max_retries=retries))
session.mount("https://", HTTPAdapter(max_retries=retries))
session.headers.update({"User-Agent": USER_AGENT})

MITRE_DOMAINS = ("attack.mitre.org", "mitre.org", "github.com/mitre")

def sha1_url(url: str) -> str:
    return hashlib.sha1(url.encode("utf-8")).hexdigest()

def guess_ext(url, ctype):
    if ctype:
        ext = mimetypes.guess_extension(ctype.split(";")[0].strip())
        if ext:
            return ext
    path = urlparse(url).path
    _, ext = os.path.splitext(path)
    return ext if ext else ".bin"

def download(url, local_path, max_size=MAX_SIZE_BYTES):
    r = session.get(url, stream=True, timeout=TIMEOUT)
    r.raise_for_status()
    total = 0
    with open(local_path, "wb") as f:
        for chunk in r.iter_content(8192):
            if not chunk:
                continue
            total += len(chunk)
            if max_size and total > max_size:
                f.close()
                os.remove(local_path)
                raise RuntimeError(f"File too large (> {max_size} bytes)")
            f.write(chunk)
    return r.headers.get("Content-Type", ""), total

def extract_text(path, content_type):
    ctype = (content_type or "").lower()
    try:
        if "pdf" in ctype or path.lower().endswith(".pdf"):
            return extract_pdf_text(path)
        elif any(x in ctype for x in ("html", "xml", "text")):
            with open(path, "rb") as f:
                data = f.read()
            text = trafilatura.extract(data)
            return text or ""
        else:
            with open(path, "rb") as f:
                data = f.read()
            text = trafilatura.extract(data)
            return text or ""
    except Exception:
        return ""

def is_mitre_link(url):
    host = urlparse(url).netloc.lower()
    return any(domain in host for domain in MITRE_DOMAINS)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def main():
    data = load_json(INPUT_FILE)
    objects = data.get("objects", [])
    print(f"[+] Loaded {len(objects)} STIX objects")

    # Load existing manifest if any
    if os.path.exists(OUT_FILE):
        with open(OUT_FILE, "r", encoding="utf-8") as f:
            manifest = json.load(f)
    else:
        manifest = []

    # Map already downloaded hashes to avoid duplicates
    known_hashes = {m.get("download_hash") for m in manifest if m.get("download_hash")}

    for obj in objects:
        if obj.get("type") != "attack-pattern":
            continue

        tid = next((r["external_id"] for r in obj.get("external_references", []) if r.get("source_name") == "mitre-attack"), None)
        tname = obj.get("name")
        if not tid:
            continue

        for ref in obj.get("external_references", []):
            url = ref.get("url")
            if not url or is_mitre_link(url):
                continue

            h = sha1_url(url)
            # If already downloaded → just associate
            if h in known_hashes:
                manifest.append({
                    "technique_id": tid,
                    "technique_name": tname,
                    "reference_url": url,
                    "download_hash": h,
                    "status": "already_downloaded"
                })
                continue

            filename = f"{datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}_{h[:10]}{guess_ext(url, None)}"
            local_path = os.path.join(DOWNLOAD_DIR, filename)

            print(f"[>] {tid}: downloading {url}")
            try:
                ctype, size = download(url, local_path)
                text = extract_text(local_path, ctype)
                entry = {
                    "technique_id": tid,
                    "technique_name": tname,
                    "source_name": ref.get("source_name"),
                    "reference_url": url,
                    "local_path": local_path,
                    "content_type": ctype,
                    "file_size": size,
                    "download_hash": h,
                    "extracted_text": text,  # truncate preview
                    "status": "ok"
                }
                manifest.append(entry)
                known_hashes.add(h)
            except Exception as e:
                manifest.append({
                    "technique_id": tid,
                    "technique_name": tname,
                    "reference_url": url,
                    "error": str(e),
                    "status": "failed",
                    "download_hash": h
                })
            time.sleep(REQUEST_DELAY)

        # Periodic save
        with open(OUT_FILE, "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2, ensure_ascii=False)

    print(f"[+] Done. Saved {len(manifest)} records → {OUT_FILE}")


if __name__ == "__main__":
    main()


[+] Loaded 24773 STIX objects
[>] T1055.011: downloading https://msdn.microsoft.com/library/windows/desktop/ms633574.aspx
[>] T1055.011: downloading https://msdn.microsoft.com/library/windows/desktop/ms633584.aspx
[>] T1055.011: downloading https://msdn.microsoft.com/library/windows/desktop/ms633591.aspx
[>] T1055.011: downloading https://www.endgame.com/blog/technical-blog/ten-process-injection-techniques-technical-survey-common-and-trending-process
[>] T1055.011: downloading https://www.malwaretech.com/2013/08/powerloader-injection-something-truly.html
[>] T1055.011: downloading https://www.welivesecurity.com/2013/03/19/gapz-and-redyms-droppers-based-on-power-loader-code/
[>] T1055.011: downloading https://msdn.microsoft.com/library/windows/desktop/ms644953.aspx
[>] T1053.005: downloading https://www.proofpoint.com/us/blog/threat-insight/serpent-no-swiping-new-backdoor-targets-french-entities-unique-attack-chain
[>] T1053.005: downloading https://blog.qualys.com/vulnerabilities-threa

Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P0' is an invalid float value


[>] T1583: downloading https://cloud.google.com/blog/topics/threat-intelligence/scandalous-external-detection-using-network-scan-data-and-automation/
[>] T1583: downloading https://threatconnect.com/blog/infrastructure-research-hunting/
[>] T1218.011: downloading https://www.cynet.com/attack-techniques-hands-on/defense-evasion-techniques/
[>] T1218.011: downloading https://www.attackify.com/blog/rundll32_execution_order/
[>] T1218.011: downloading https://www.stormshield.com/news/poweliks-command-line-confusion/
[>] T1218.011: downloading https://github.com/gtworek/PSBits/tree/master/NoRunDll
[>] T1218.011: downloading https://lolbas-project.github.io/lolbas/Libraries/Ieframe/
[>] T1218.011: downloading https://lolbas-project.github.io/lolbas/Libraries/Zipfldr/
[>] T1218.011: downloading https://www.trendmicro.de/cloud-content/us/pdfs/security-intelligence/white-papers/wp-cpl-malware.pdf
[>] T1613: downloading https://docs.docker.com/engine/api/v1.41/
[>] T1613: downloading https://kub

Cannot set gray non-stroke color because /'P44' is an invalid float value
Cannot set gray non-stroke color because /'P65' is an invalid float value
Cannot set gray non-stroke color because /'P84' is an invalid float value
Cannot set gray non-stroke color because /'P101' is an invalid float value
Cannot set gray non-stroke color because /'P113' is an invalid float value
Cannot set gray non-stroke color because /'P131' is an invalid float value
Cannot set gray non-stroke color because /'P147' is an invalid float value
Cannot set gray non-stroke color because /'P163' is an invalid float value
Cannot set gray non-stroke color because /'P181' is an invalid float value
Cannot set gray non-stroke color because /'P197' is an invalid float value
Cannot set gray non-stroke color because /'P214' is an invalid float value
Cannot set gray non-stroke color because /'P233' is an invalid float value
Cannot set gray non-stroke color because /'P250' is an invalid float value
Cannot set gray non-stroke c

[>] T1666: downloading https://techcommunity.microsoft.com/t5/microsoft-365-defender-blog/hunt-for-compromised-azure-subscriptions-using-microsoft/ba-p/3607121
[>] T1666: downloading https://learn.microsoft.com/en-us/azure/cloud-adoption-framework/ready/azure-setup-guide/organize-resources
[>] T1666: downloading https://www.microsoft.com/en-us/security/blog/2023/09/14/peach-sandstorm-password-spray-campaigns-enable-intelligence-collection-at-high-value-targets/
[>] T1564.008: downloading https://support.apple.com/guide/mail/use-rules-to-manage-emails-you-receive-mlhlp1017/mac
[>] T1564.008: downloading https://www.microsoft.com/security/blog/2021/06/14/behind-the-scenes-of-business-email-compromise-using-cross-domain-threat-data-to-disrupt-a-large-bec-infrastructure/
[>] T1564.008: downloading https://learn.microsoft.com/en-us/exchange/security-and-compliance/mail-flow-rules/mail-flow-rules
[>] T1564.008: downloading https://support.microsoft.com/en-us/office/manage-email-messages-by-u

...

[>] T1003.004: downloading https://ired.team/offensive-security/credential-access-and-credential-dumping/dumping-lsa-secrets
[>] T1003.004: downloading https://docs.microsoft.com/en-us/windows-server/identity/securing-privileged-access/securing-privileged-access-reference-material?redirectedfrom=MSDN
[>] T1003.004: downloading https://www.passcape.com/index.php?section=docsys&cmd=details&id=23
[>] T1013: downloading http://msdn.microsoft.com/en-us/library/dd183341
[>] T1013: downloading https://www.defcon.org/images/defcon-22/dc-22-presentations/Bloxham/DEFCON-22-Brady-Bloxham-Windows-API-Abuse-UPDATED.pdf
[>] T1600: downloading https://blogs.cisco.com/security/evolution-of-attacks-on-cisco-ios-devices
[>] T1606.002: downloading https://blogs.microsoft.com/on-the-issues/2020/12/13/customers-protect-nation-state-cyberattacks/
[>] T1606.002: downloading https://docs.microsoft.com/en-us/azure/active-directory/develop/active-directory-configurable-token-lifetimes
[>] T1606.002: downloading

Cannot set gray stroke color because /'P0' is an invalid float value


[>] T1542.005: downloading https://tools.cisco.com/security/center/resources/integrity_assurance.html#35
[>] T1542.005: downloading https://tools.cisco.com/security/center/resources/integrity_assurance.html#7
[>] T1542.005: downloading https://tools.cisco.com/security/center/resources/integrity_assurance.html#13
[>] T1542.005: downloading https://tools.cisco.com/security/center/resources/integrity_assurance.html#23
[>] T1542.005: downloading https://tools.cisco.com/security/center/resources/integrity_assurance.html#26
[>] T1543.003: downloading https://docs.microsoft.com/windows/security/threat-protection/use-windows-event-forwarding-to-assist-in-intrusion-detection
[>] T1543.003: downloading https://www.welivesecurity.com/wp-content/uploads/2020/06/ESET_InvisiMole.pdf
[>] T1543.003: downloading https://www.sans.org/blog/red-team-tactics-hiding-windows-services/
[>] T1543.003: downloading https://www.sans.org/blog/defense-spotlight-finding-hidden-windows-services/
[>] T1543.003: downlo

Cannot set gray stroke color because /'P0' is an invalid float value
Cannot set gray stroke color because /'P1' is an invalid float value
Cannot set gray stroke color because /'P2' is an invalid float value


[>] T1082: downloading https://docs.aws.amazon.com/cli/latest/reference/ssm/describe-instance-information.html
[>] T1082: downloading https://cloud.google.com/compute/docs/reference/rest/v1/instances
[>] T1082: downloading https://www.varonis.com/blog/vmware-esxi-in-the-line-of-ransomware-fire
[>] T1082: downloading https://docs.microsoft.com/en-us/rest/api/compute/virtualmachines/get
[>] T1082: downloading https://www.sentinelone.com/blog/trail-osx-fairytale-adware-playing-malware/
[>] T1071: downloading https://arxiv.org/ftp/arxiv/papers/1408/1408.1136.pdf
[>] T1574.014: downloading https://pentestlaboratories.com/2020/05/26/appdomainmanager-injection-and-detection/
[>] T1574.014: downloading https://learn.microsoft.com/dotnet/framework/app-domains/application-domains
[>] T1574.014: downloading https://www.pwc.com/gx/en/issues/cybersecurity/cyber-threat-intelligence/yellow-liderc-ships-its-scripts-delivers-imaploader-malware.html
[>] T1574.014: downloading https://www.rapid7.com/blog

In [2]:
#!/usr/bin/env python3
"""
analyze_external_reports.py

Reads external_links_to_ttps.json (produced by the scraper)
and builds a cross-reference summary of:
 - unique reports (by download_hash)
 - list of techniques that reference each report
Then prints summary statistics.
"""

import json
import os
from urllib.parse import urlparse
from collections import defaultdict, Counter
from statistics import mean

INPUT_FILE = "external_links_to_ttps.json"
OUT_CROSSREF = "external_reports_crossref.json"

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def get_domain(url):
    try:
        return urlparse(url).netloc.lower()
    except Exception:
        return "unknown"

def main():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ {INPUT_FILE} not found. Run your scraper first.")
        return

    print(f"[+] Loading manifest: {INPUT_FILE}")
    data = load_json(INPUT_FILE)
    print(f"[+] Loaded {len(data)} total associations")

    # --- Cross-reference dictionary ---
    reports = {}  # hash → dict with metadata + list of TTPs
    domain_counter = Counter()
    total_associations = 0

    for entry in data:
        h = entry.get("download_hash")
        if not h:
            continue
        tid = entry.get("technique_id")
        tname = entry.get("technique_name")
        url = entry.get("reference_url")
        status = entry.get("status", "")
        domain = get_domain(url)
        domain_counter[domain] += 1

        if h not in reports:
            reports[h] = {
                "reference_url": url,
                "domain": domain,
                "local_path": entry.get("local_path"),
                "content_type": entry.get("content_type"),
                "file_size": entry.get("file_size"),
                "status": entry.get("status"),
                "techniques": []
            }

        # Append new TTP reference (avoid duplicates)
        if tid and tid not in [t["id"] for t in reports[h]["techniques"]]:
            reports[h]["techniques"].append({"id": tid, "name": tname})
            total_associations += 1
        # --- Count unique TTPs across all reports ---
        all_ttps = []
        for r in reports.values():
            for t in r["techniques"]:
                all_ttps.append(t["id"])

        total_ttps_used = len(all_ttps)                # counting all usages
        unique_ttps_used = len(set(all_ttps))          # counting unique TTPs
    # --- Save cross-reference JSON ---
    with open(OUT_CROSSREF, "w", encoding="utf-8") as f:
        json.dump(list(reports.values()), f, indent=2, ensure_ascii=False)

    # --- Compute statistics ---
    num_reports = len(reports)
    ttps_per_report = [len(v["techniques"]) for v in reports.values()]

    avg_ttps = mean(ttps_per_report) if ttps_per_report else 0
    max_ttps = max(ttps_per_report) if ttps_per_report else 0


    print("\n📊 === Statistics ===")
    print(f"Unique reports: {num_reports}")
    print(f"Total TTP ↔ report associations: {total_associations}")
    print(f"Total TTPs used (with duplicates): {total_ttps_used}")
    print(f"Total unique TTPs used: {unique_ttps_used}")
    print(f"Average TTPs per report: {avg_ttps:.2f}")
    print(f"Max TTPs in one report: {max_ttps}")

    print(f"\nTop 100 domains:")
    for dom, cnt in domain_counter.most_common(100):
        print(f"  {dom:40} {cnt}")

    print(f"\nTop 10 most-referenced reports:")
    top_reports = sorted(
        reports.values(),
        key=lambda r: len(r["techniques"]),
        reverse=True
    )[:10]
    for r in top_reports:
        print(f"  [{len(r['techniques'])} TTPs] {r['reference_url']}")

    print(f"\n[+] Cross-reference JSON saved → {OUT_CROSSREF}")
    # --- Build a sorted list of unique TTPs (id + name) ---
    unique_ttp_objects = {}
    for r in reports.values():
        for t in r["techniques"]:
            unique_ttp_objects[t["id"]] = t["name"]

    # Sort by TTP ID
    sorted_ttps = sorted(unique_ttp_objects.items(), key=lambda x: x[0])
    print("\n🧩 All unique TTPs used:")
    for tid, tname in sorted_ttps:
        print(f"  {tid}: {tname}")

if __name__ == "__main__":
    main()


[+] Loading manifest: external_links_to_ttps.json
[+] Loaded 3367 total associations

📊 === Statistics ===
Unique reports: 2281
Total TTP ↔ report associations: 3363
Total TTPs used (with duplicates): 3363
Total unique TTPs used: 797
Average TTPs per report: 1.47
Max TTPs in one report: 40

Top 100 domains:
  docs.microsoft.com                       198
  github.com                               121
  msdn.microsoft.com                       104
  web.archive.org                          88
  technet.microsoft.com                    83
  learn.microsoft.com                      60
  cloud.google.com                         57
  unit42.paloaltonetworks.com              54
  www.welivesecurity.com                   52
  www.fireeye.com                          52
  en.wikipedia.org                         51
  www.microsoft.com                        49
  www.mandiant.com                         45
  arxiv.org                                45
  docs.aws.amazon.com                      4